# Chapter 3 Lab — Agent Anatomy

**Book:** *Agentic AI: Build, Ship, Portfolio*
**Chapter 3:** Agent Anatomy — The Four Pillars

---

In this lab you will build every structural component of an LLM-based agent from
scratch so that you can see — and break — each pillar independently.

| Section | What You Build |
|---------|---------------|
| 1 | `Perception`, `Memory`, `Reasoning`, `Action` component classes |
| 2 | `MinimalAgent` that wires all four pillars into a working loop |
| 3 | An **Agent Blueprint Analyzer** that audits any AI system description |
| 4 | **Failure-mode diagnosis** — deliberately break pillars and observe |
| 5 | Exercises — pillar audit, vector memory, dynamic tool belt |

> **Prerequisites:** An OpenAI API key stored in a `.env` file or as the
> environment variable `OPENAI_API_KEY`.

## Setup

In [ ]:
%pip install openai python-dotenv --quiet

In [ ]:
from __future__ import annotations

import json
import math
import os
import textwrap
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Callable

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()                       # reads OPENAI_API_KEY from env
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# Quick connectivity check
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say OK"}],
    max_tokens=3,
)
print(f"Model: {MODEL}  |  Response: {resp.choices[0].message.content}")

---
## Section 1 — The Four Pillars in Code

We define an `AgentComponent` abstract base class, then implement each pillar as
a concrete subclass. Each pillar is intentionally small so you can see the
minimal viable version before we wire them together.

> **Chapter 3 recap:** Every agent is built from four pillars — *Perception*,
> *Memory*, *Reasoning*, and *Action*. The LLM is the reasoning core; the other
> three pillars constrain it and connect it to the world.

In [ ]:
# ── Base class ────────────────────────────────────────────────────────

class AgentComponent(ABC):
    """Abstract base for every agent pillar."""

    name: str  # human-readable label

    @abstractmethod
    def __call__(self, *args, **kwargs) -> Any:
        """Execute the component's primary responsibility."""
        ...

    def describe(self) -> str:
        """One-line summary used for diagnostics."""
        return f"{self.__class__.__name__}: {getattr(self, 'name', '?')}"


print("AgentComponent base class ready.")

### Pillar 1 — Perception

Perception transforms **raw input** into **structured information** the reasoning
core can consume. Our minimal version strips whitespace, detects a simple intent
keyword, and wraps the result in a typed dict.

In [ ]:
class Perception(AgentComponent):
    """
    Pillar 1 — Perception
    Takes raw user input, normalizes it, detects intent keywords,
    and packages the result as a structured observation dict.
    """

    name = "Perception"

    # Simple keyword-based intent hints (a real system would use a classifier)
    INTENT_KEYWORDS: dict[str, list[str]] = {
        "calculate":  ["calculate", "compute", "math", "sum", "add", "multiply", "divide", "subtract"],
        "string_op":  ["reverse", "uppercase", "lowercase", "length", "count"],
        "question":   ["what", "why", "how", "explain", "describe", "tell me"],
        "greeting":   ["hello", "hi", "hey", "good morning"],
    }

    def __call__(self, raw_input: str) -> dict:
        """Parse raw input into a structured observation."""
        cleaned = raw_input.strip()
        lower = cleaned.lower()

        # Detect intent hint
        detected_intent = "general"
        for intent, keywords in self.INTENT_KEYWORDS.items():
            if any(kw in lower for kw in keywords):
                detected_intent = intent
                break

        observation = {
            "raw":       raw_input,
            "cleaned":   cleaned,
            "intent":    detected_intent,
            "timestamp": datetime.now().isoformat(),
        }
        return observation


# ── Quick test ──
percept = Perception()
obs = percept("  Calculate 42 * 7 please  ")
print(json.dumps(obs, indent=2))

### Pillar 2 — Memory

Buffer memory stores the rolling conversation history. Our implementation
supports `add`, `get_history`, and `clear`, with an optional sliding-window
limit so we never blow the context window.

In [ ]:
class Memory(AgentComponent):
    """
    Pillar 2 — Memory (buffer / short-term)
    A sliding-window conversation store.
    """

    name = "Memory"

    def __init__(self, max_turns: int = 20):
        self.max_turns = max_turns
        self._buffer: list[dict[str, str]] = []

    # --- core API ---
    def add(self, role: str, content: str) -> None:
        """Append a message and enforce the window limit."""
        self._buffer.append({"role": role, "content": content})
        # Slide: keep system messages + last N user/assistant turns
        non_system = [m for m in self._buffer if m["role"] != "system"]
        if len(non_system) > self.max_turns:
            # Drop the oldest non-system message
            for i, m in enumerate(self._buffer):
                if m["role"] != "system":
                    self._buffer.pop(i)
                    break

    def get_history(self) -> list[dict[str, str]]:
        """Return the full buffer (suitable for the messages= parameter)."""
        return list(self._buffer)

    def clear(self) -> None:
        self._buffer.clear()

    def __call__(self, role: str, content: str) -> list[dict[str, str]]:
        """Add a message and return current history."""
        self.add(role, content)
        return self.get_history()

    def __len__(self) -> int:
        return len(self._buffer)


# ── Quick test ──
mem = Memory(max_turns=4)
mem.add("system", "You are a helpful assistant.")
mem.add("user", "Hello")
mem.add("assistant", "Hi there!")
mem.add("user", "What is 2+2?")
mem.add("assistant", "4")
mem.add("user", "And 3+3?")  # this should trigger window slide

print(f"Buffer length: {len(mem)}")
for m in mem.get_history():
    print(f"  [{m['role']:>9}] {m['content']}")

### Pillar 3 — Reasoning

The reasoning core wraps the OpenAI Chat Completions API. It accepts the
current message history (from Memory) plus optional tool definitions (from
Action) and returns the model's response — which may include tool calls.

In [ ]:
class Reasoning(AgentComponent):
    """
    Pillar 3 — Reasoning
    Wraps the LLM (OpenAI Chat Completions) as the decision-making core.
    """

    name = "Reasoning"

    def __init__(self, client: OpenAI, model: str = MODEL):
        self.client = client
        self.model = model

    def __call__(
        self,
        messages: list[dict],
        tools: list[dict] | None = None,
        temperature: float = 0.3,
    ):
        """
        Send messages (+ optional tool defs) to the LLM and return the
        full response message object.
        """
        kwargs: dict[str, Any] = dict(
            model=self.model,
            messages=messages,
            temperature=temperature,
        )
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"

        response = self.client.chat.completions.create(**kwargs)
        return response.choices[0].message


# ── Quick test ──
reasoning = Reasoning(client)
test_msg = reasoning([{"role": "user", "content": "What is 7 * 6?"}])
print(f"Reasoning response: {test_msg.content}")

### Pillar 4 — Action

The Action layer is a **tool registry**. You register Python functions as tools,
each with an OpenAI-compatible JSON schema. When the reasoning core emits a tool
call, the Action layer validates and executes it, then returns the result.

In [ ]:
class Action(AgentComponent):
    """
    Pillar 4 — Action (Tool Belt)
    Registry of callable tools with OpenAI-compatible schemas.
    """

    name = "Action"

    def __init__(self):
        self._tools: dict[str, Callable] = {}
        self._schemas: list[dict] = []

    def register(self, func: Callable, schema: dict) -> None:
        """Register a tool function with its OpenAI tool schema."""
        name = schema["function"]["name"]
        self._tools[name] = func
        self._schemas.append(schema)
        print(f"  Registered tool: {name}")

    def get_schemas(self) -> list[dict]:
        """Return all tool schemas (for the tools= parameter)."""
        return list(self._schemas)

    def execute(self, tool_name: str, arguments: dict) -> str:
        """Validate, execute a tool, and return the string result."""
        if tool_name not in self._tools:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        try:
            result = self._tools[tool_name](**arguments)
            return json.dumps({"result": result})
        except Exception as e:
            return json.dumps({"error": str(e)})

    def __call__(self, tool_name: str, arguments: dict) -> str:
        return self.execute(tool_name, arguments)


# ── Register a few simple tools ─────────────────────────────────────

action = Action()

# Tool 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely."""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return f"Invalid characters in expression: {expression}"
    try:
        result = eval(expression)  # safe because we whitelisted chars
        return str(result)
    except Exception as e:
        return f"Error: {e}"

action.register(calculator, {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Evaluate a mathematical expression. Supports +, -, *, /, parentheses.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The math expression to evaluate, e.g. '(42 * 7) + 3'"
                }
            },
            "required": ["expression"]
        }
    }
})

# Tool 2: String operations
def string_tool(text: str, operation: str) -> str:
    """Perform a string operation."""
    ops = {
        "reverse":   lambda t: t[::-1],
        "uppercase": lambda t: t.upper(),
        "lowercase": lambda t: t.lower(),
        "length":    lambda t: str(len(t)),
        "wordcount": lambda t: str(len(t.split())),
    }
    if operation not in ops:
        return f"Unknown operation '{operation}'. Choose from: {list(ops.keys())}"
    return ops[operation](text)

action.register(string_tool, {
    "type": "function",
    "function": {
        "name": "string_tool",
        "description": "Perform a string operation: reverse, uppercase, lowercase, length, or wordcount.",
        "parameters": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The input text"},
                "operation": {
                    "type": "string",
                    "enum": ["reverse", "uppercase", "lowercase", "length", "wordcount"],
                    "description": "Which operation to perform"
                }
            },
            "required": ["text", "operation"]
        }
    }
})

# Tool 3: Current timestamp
def get_timestamp() -> str:
    """Return the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

action.register(get_timestamp, {
    "type": "function",
    "function": {
        "name": "get_timestamp",
        "description": "Return the current date and time.",
        "parameters": {"type": "object", "properties": {}}
    }
})

# Quick test
print("\nDirect tool call:", action("calculator", {"expression": "42 * 7"}))

---
## Section 2 — Building a Minimal Agent

Now we wire all four pillars into a `MinimalAgent` class that implements the
**agent loop** from Chapter 3:

1. **Perception** receives input and produces a structured observation.
2. **Memory** retrieval — assemble conversation history.
3. **Reasoning** deliberates — the LLM produces text and/or tool calls.
4. **Action** executes any tool calls; results feed back to step 3.
5. **Memory** update — store the new turn.
6. **Loop evaluation** — repeat if there are pending tool calls.

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
    You are a helpful assistant with access to tools.
    When the user asks you to calculate something, use the calculator tool.
    When the user asks about string operations, use the string_tool.
    When the user asks for the current time, use get_timestamp.
    Always use tools when available rather than guessing answers.
    After receiving tool results, provide a clear final answer.
""")


class MinimalAgent:
    """
    A minimal agent that wires together all four pillars:
    Perception -> Memory -> Reasoning -> Action -> (loop)
    """

    def __init__(
        self,
        perception: Perception,
        memory: Memory,
        reasoning: Reasoning,
        action: Action,
        system_prompt: str = SYSTEM_PROMPT,
        max_iterations: int = 5,
        verbose: bool = True,
    ):
        self.perception = perception
        self.memory = memory
        self.reasoning = reasoning
        self.action = action
        self.max_iterations = max_iterations
        self.verbose = verbose

        # Seed memory with the system prompt
        self.memory.clear()
        self.memory.add("system", system_prompt)

    def _log(self, step: str, detail: str) -> None:
        if self.verbose:
            print(f"  [{step:>12}] {detail}")

    def run(self, user_input: str) -> str:
        """Execute the full agent loop for one user turn."""
        print(f"\n{'='*60}")
        print(f"USER: {user_input}")
        print(f"{'='*60}")

        # ── Step 1: Perception ───────────────────────────────────
        observation = self.perception(user_input)
        self._log("Perception", f"intent={observation['intent']}")

        # ── Step 2: Memory — store user message ──────────────────
        self.memory.add("user", observation["cleaned"])
        self._log("Memory", f"buffer size = {len(self.memory)}")

        # ── Steps 3-6: Reasoning + Action loop ───────────────────
        for iteration in range(1, self.max_iterations + 1):
            self._log("Reasoning", f"iteration {iteration}")

            # Get LLM response
            messages = self.memory.get_history()
            tools = self.action.get_schemas() if self.action._tools else None
            response = self.reasoning(messages, tools=tools)

            # If the response has tool calls, execute them
            if response.tool_calls:
                # Store the assistant message with tool calls
                self.memory.add("assistant", response.content or "")
                # We need to store the raw message for the API
                # Replace last assistant message with the full object
                self.memory._buffer[-1] = response.model_dump(
                    exclude_none=True, exclude_unset=True
                )

                for tc in response.tool_calls:
                    fn_name = tc.function.name
                    fn_args = json.loads(tc.function.arguments)
                    self._log("Action", f"call {fn_name}({fn_args})")

                    result = self.action.execute(fn_name, fn_args)
                    self._log("Action", f"result = {result}")

                    # Feed tool result back into memory
                    self.memory._buffer.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": result,
                    })
            else:
                # No tool calls — the LLM produced a final answer
                final = response.content or "(no response)"
                self.memory.add("assistant", final)
                self._log("Done", f"iterations={iteration}")
                print(f"\nAGENT: {final}")
                return final

        # Safety net: max iterations reached
        fallback = "I reached the maximum number of reasoning steps. Here is what I have so far."
        self.memory.add("assistant", fallback)
        print(f"\nAGENT: {fallback}")
        return fallback


print("MinimalAgent class defined.")

### Run the agent on a few tasks

Watch the step-by-step output to see each pillar activate in sequence.

In [ ]:
# Instantiate all four pillars
perception = Perception()
memory     = Memory(max_turns=20)
reasoning  = Reasoning(client)
# 'action' was already built above with tools registered

agent = MinimalAgent(perception, memory, reasoning, action)

# Task 1: Math — exercises the calculator tool
agent.run("What is (123 * 456) + 789?")

# Task 2: String ops — exercises the string_tool
agent.run("Reverse the string 'Agent Anatomy' and tell me its length")

# Task 3: Multi-step — the agent should remember the earlier result
agent.run("Take the math result from earlier and divide it by 3")

---
## Section 3 — Agent Blueprint Analyzer

The chapter's main project: a tool that takes a **natural-language description
of an AI system** and uses the LLM to analyze which of the four pillars are
present, missing, or weak. It produces a structured **report card**.

In [ ]:
ANALYZER_PROMPT = textwrap.dedent("""\
    You are an AI architecture reviewer specializing in agentic systems.

    The user will describe an AI system. Analyze it against the Four Pillars
    of agent anatomy:

    1. **Perception** — input parsing, intent classification, multimodal
       ingestion, context enrichment.
    2. **Memory** — buffer (short-term), vector (long-term semantic),
       episodic (experiential). Does the system remember across turns?
       Across sessions?
    3. **Reasoning** — LLM-based decision making, planning strategy
       (reactive vs. deliberative), situation assessment, plan generation.
    4. **Action** — tool belt, validation/sandboxing, result processing,
       audit logging. What can the system actually DO?

    For each pillar, provide:
    - **Status**: Present / Partial / Missing
    - **Evidence**: Quote or reference what the description says
    - **Risk**: What could go wrong because of the current state
    - **Recommendation**: One concrete improvement

    End with an overall **Architecture Grade** (A through F) and a one-paragraph
    summary.

    Format your response as clean Markdown.
""")


def analyze_blueprint(description: str) -> str:
    """
    Send a system description to the LLM for pillar-by-pillar analysis.
    Returns the Markdown report card.
    """
    messages = [
        {"role": "system", "content": ANALYZER_PROMPT},
        {"role": "user",   "content": description},
    ]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.4,
    )
    return response.choices[0].message.content


print("analyze_blueprint() defined.")

In [ ]:
# ── Test 1: A well-designed system ────────────────────────────────────

good_system = """\
Our customer-service agent receives tickets via email and Slack. Incoming
messages pass through a lightweight BERT classifier that detects intent
(refund, status inquiry, complaint, general question) and extracts entities
(order ID, product name). The classified input is fed to GPT-4o along with
the last 10 conversation turns (buffer memory) and relevant knowledge-base
articles retrieved from a Pinecone vector store. The agent can call four
tools: look_up_order, process_refund, escalate_to_human, and send_email.
All tool calls are validated against the user's permission level, logged to
a Datadog audit trail, and rate-limited to 5 refunds per hour. The agent
uses a deliberative planning loop: it generates a plan, asks the user for
confirmation on irreversible actions, then executes.
"""

from IPython.display import Markdown, display
display(Markdown(analyze_blueprint(good_system)))

In [ ]:
# ── Test 2: A poorly-designed system (the Acme refund agent from Ch.3) ──

bad_system = """\
We built a customer-service chatbot. It uses GPT-4 with a system prompt that
says 'You are a helpful customer-service agent for Acme Corp. When a customer
requests a refund, process it.' We connected a process_refund tool. We tested
it on ten sample conversations and it worked. There is no conversation history
stored between turns. There is no input classification. There is no
confirmation step before processing refunds. There is no logging.
"""

display(Markdown(analyze_blueprint(bad_system)))

---
## Section 4 — Failure Mode Diagnosis

What happens when we deliberately **remove or break** each pillar? This section
builds the same agent but disables one component at a time, demonstrating that
every pillar is a *structural, load-bearing wall* — not an optional accessory.

We will test the same query — `"What is (42 * 7) + 10? Also reverse 'hello world'."` —
under four conditions:
1. Full agent (baseline)
2. No memory (fresh buffer every turn)
3. No tools (Action layer is empty)
4. No perception (raw input passed directly, no structuring)

In [ ]:
TEST_QUERY = "What is (42 * 7) + 10? Also reverse 'hello world'."

results = {}

# ── Scenario 1: Full agent (baseline) ────────────────────────────────
print("\n" + "="*70)
print("SCENARIO 1: FULL AGENT (BASELINE)")
print("="*70)

agent_full = MinimalAgent(
    Perception(), Memory(max_turns=20), Reasoning(client), action,
    verbose=True
)
results["full"] = agent_full.run(TEST_QUERY)

In [ ]:
# ── Scenario 2: No memory (amnesiac agent) ───────────────────────────
# We subclass Memory so that get_history always returns only the system
# prompt + the current message (no prior turns).

print("\n" + "="*70)
print("SCENARIO 2: NO MEMORY (amnesiac — forgets everything)")
print("="*70)

class AmnesiacMemory(Memory):
    """A broken memory that never retains prior turns."""
    def get_history(self) -> list[dict[str, str]]:
        # Return only system message + the last user message
        system_msgs = [m for m in self._buffer if m.get("role") == "system"]
        last_user = [m for m in self._buffer if m.get("role") == "user"][-1:]
        return system_msgs + last_user

agent_no_mem = MinimalAgent(
    Perception(), AmnesiacMemory(max_turns=20), Reasoning(client), action,
    verbose=True
)
# Ask two questions — the second references the first
agent_no_mem.run("What is 100 + 200?")
results["no_memory"] = agent_no_mem.run("Double that previous result")
# The agent should fail to recall "300" because memory is broken

In [ ]:
# ── Scenario 3: No tools (brain in a jar) ────────────────────────────
# The agent has reasoning but no way to act on the world.

print("\n" + "="*70)
print("SCENARIO 3: NO TOOLS (brain in a jar — can talk but cannot act)")
print("="*70)

empty_action = Action()  # no tools registered

agent_no_tools = MinimalAgent(
    Perception(), Memory(max_turns=20), Reasoning(client), empty_action,
    verbose=True
)
results["no_tools"] = agent_no_tools.run(TEST_QUERY)
# The agent will try to answer the math without a calculator — watch for errors

In [ ]:
# ── Scenario 4: No perception (raw, unstructured input) ──────────────
# We replace Perception with a pass-through that does no processing.

print("\n" + "="*70)
print("SCENARIO 4: NO PERCEPTION (raw input, no structuring)")
print("="*70)

class NullPerception(Perception):
    """Broken perception — passes raw input with no processing."""
    def __call__(self, raw_input: str) -> dict:
        return {
            "raw":       raw_input,
            "cleaned":   raw_input,    # no cleaning
            "intent":    "unknown",    # no classification
            "timestamp": datetime.now().isoformat(),
        }

# Send messy input — extra whitespace, mixed case, no clear structure
messy_input = "   \n\n   calculate   42 *    7   + 10  AND ALSO reverse 'hello world' please!!!\n\n  "

agent_no_percept = MinimalAgent(
    NullPerception(), Memory(max_turns=20), Reasoning(client), action,
    verbose=True
)
results["no_perception"] = agent_no_percept.run(messy_input)

In [ ]:
# ── Comparison summary ────────────────────────────────────────────────

print("\n" + "="*70)
print("FAILURE MODE COMPARISON")
print("="*70)

comparison_prompt = textwrap.dedent("""\
    You are an AI architecture instructor. Compare these four agent responses
    to the same query and explain what each failure mode teaches us about the
    importance of the missing pillar. Be specific and reference the actual
    outputs.

    **Query:** {query}

    **Scenario 1 — Full Agent (baseline):**
    {full}

    **Scenario 2 — No Memory (asked '100+200' then 'double that'):**
    {no_memory}

    **Scenario 3 — No Tools:**
    {no_tools}

    **Scenario 4 — No Perception (messy raw input):**
    {no_perception}

    Format as a Markdown table with columns: Scenario | Missing Pillar |
    Observed Behavior | Lesson Learned. Then add a brief conclusion.
""").format(query=TEST_QUERY, **results)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": comparison_prompt}],
    temperature=0.3,
)
display(Markdown(resp.choices[0].message.content))

---
## Section 5 — Exercises

### Exercise 1 (Conceptual): Pillar Audit of Real Systems

Pick **three** real AI systems from the list below. For each one, write a
short description (3-5 sentences) of what you think their Perception, Memory,
Reasoning, and Action pillars look like. Then run them through
`analyze_blueprint()` and compare the LLM's analysis to yours.

Systems to choose from:
- GitHub Copilot
- ChatGPT with code interpreter
- Tesla Autopilot
- Alexa / Siri voice assistant
- Autonomous trading bot
- Medical imaging diagnostic AI

In [ ]:
# ── Exercise 1: Pillar audit ──────────────────────────────────────────
# Replace the placeholder descriptions below with your own analysis,
# then run this cell to compare with the LLM's review.

my_systems = {
    "GitHub Copilot": """\
        GitHub Copilot receives code context from the IDE (current file,
        open tabs, cursor position). It uses an LLM to predict the next
        lines of code. It has no persistent memory across sessions. Its
        primary action is inserting code suggestions into the editor.
    """,

    "ChatGPT with Code Interpreter": """\
        ChatGPT receives user messages (text, images, files). It maintains
        a conversation buffer within a session. It can execute Python code
        in a sandboxed environment, read/write files, and generate images.
        It uses GPT-4 for reasoning with a deliberative planning approach.
    """,

    "Alexa Voice Assistant": """\
        Alexa uses a speech-to-text perception layer to convert audio into
        text. It classifies intents using a dedicated NLU model. It has
        short-term buffer memory within a conversation but limited long-term
        memory. It can control smart home devices, play music, and call APIs.
    """,
}

for name, description in my_systems.items():
    print(f"\n{'='*60}")
    print(f"SYSTEM: {name}")
    print(f"{'='*60}")
    report = analyze_blueprint(description)
    display(Markdown(report))

### Exercise 2 (Coding): Add ChromaDB Vector Memory

Extend the `Memory` class to include a **vector memory** store alongside the
buffer. Use ChromaDB as the vector database. The enhanced memory should:

1. Store every assistant response as an embedding in ChromaDB.
2. On each new user query, retrieve the top-3 most relevant past responses.
3. Inject retrieved context into the prompt as a "relevant past context" section.

Starter code is below — fill in the `TODO` sections.

In [ ]:
# ── Exercise 2: Vector memory with ChromaDB ──────────────────────────
# Uncomment the pip install and fill in the TODOs.
#
# %pip install chromadb --quiet

# import chromadb
#
# class VectorMemory(Memory):
#     """Buffer memory + ChromaDB vector memory."""
#
#     def __init__(self, max_turns: int = 20, collection_name: str = "agent_memory"):
#         super().__init__(max_turns)
#         self.chroma_client = chromadb.Client()
#         self.collection = self.chroma_client.get_or_create_collection(
#             name=collection_name
#         )
#         self._doc_counter = 0
#
#     def add(self, role: str, content: str) -> None:
#         """Store in buffer AND in vector DB (for assistant messages)."""
#         super().add(role, content)
#
#         # TODO: If role == "assistant", also store the content in ChromaDB.
#         # Hint: use self.collection.add(
#         #     documents=[content],
#         #     ids=[f"doc_{self._doc_counter}"],
#         # )
#         # Don't forget to increment self._doc_counter.
#         pass
#
#     def retrieve_relevant(self, query: str, n_results: int = 3) -> list[str]:
#         """Retrieve the most relevant past responses for a query."""
#         # TODO: Use self.collection.query() to find similar documents.
#         # Return a list of document strings.
#         # Handle the case where the collection is empty.
#         pass
#
#     def get_history_with_context(self, current_query: str) -> list[dict[str, str]]:
#         """Return buffer history with injected vector-memory context."""
#         history = self.get_history()
#
#         # TODO: Call retrieve_relevant() and inject the results as a
#         # system message like:
#         # "Relevant context from past conversations:\n- ...\n- ..."
#         # Insert it right after the system prompt in the history list.
#
#         return history
#
#
# # Test your implementation:
# # vec_mem = VectorMemory()
# # vec_mem.add("system", "You are helpful.")
# # vec_mem.add("user", "What is Python?")
# # vec_mem.add("assistant", "Python is a high-level programming language.")
# # vec_mem.add("user", "What about JavaScript?")
# # vec_mem.add("assistant", "JavaScript is a web programming language.")
# # relevant = vec_mem.retrieve_relevant("Tell me about programming languages")
# # print("Retrieved:", relevant)

print("Exercise 2: Fill in the TODOs above and uncomment to test.")

### Exercise 3 (Design): Dynamic Tool Belt

As described in Chapter 3, production agents often use **dynamic tool belts**
where the available tools change based on context (user permissions, workflow
stage, etc.).

**Your task:** Design and implement a `DynamicAction` class that extends `Action`
with the following capabilities:

1. **Role-based access:** Tools can be tagged with required roles (e.g., "admin",
   "support"). The `get_schemas()` method accepts a `user_role` parameter and
   only returns tools the user is authorized to use.
2. **Workflow gates:** Some tools are only available after a precondition is met
   (e.g., `deploy` is only available after `run_tests` returns success).
3. **Usage tracking:** Log every tool invocation with timestamp, tool name,
   arguments, and result for an audit trail.

Starter code is below.

In [ ]:
# ── Exercise 3: Dynamic tool belt ─────────────────────────────────────

# @dataclass
# class ToolMeta:
#     """Metadata for a registered tool."""
#     func: Callable
#     schema: dict
#     required_role: str = "any"          # "any", "admin", "support", etc.
#     precondition: str | None = None     # name of tool that must succeed first
#
#
# class DynamicAction(Action):
#     """Action layer with role-based access, workflow gates, and audit logging."""
#
#     def __init__(self):
#         super().__init__()
#         self._meta: dict[str, ToolMeta] = {}
#         self._audit_log: list[dict] = []
#         self._completed_tools: set[str] = set()  # tracks which tools have run
#
#     def register(self, func: Callable, schema: dict,
#                  required_role: str = "any",
#                  precondition: str | None = None) -> None:
#         """Register a tool with role and precondition metadata."""
#         name = schema["function"]["name"]
#         # TODO: Store in self._meta AND call super().register()
#         pass
#
#     def get_schemas(self, user_role: str = "any") -> list[dict]:
#         """Return only the tool schemas the user is authorized to use."""
#         # TODO: Filter by role AND by precondition (check self._completed_tools)
#         pass
#
#     def execute(self, tool_name: str, arguments: dict) -> str:
#         """Execute with audit logging and workflow tracking."""
#         # TODO:
#         # 1. Call super().execute()
#         # 2. Log to self._audit_log with timestamp, tool_name, arguments, result
#         # 3. Add tool_name to self._completed_tools on success
#         # 4. Return the result
#         pass
#
#     def get_audit_log(self) -> list[dict]:
#         """Return the full audit trail."""
#         return list(self._audit_log)
#
#
# # Test scenario:
# # dyn = DynamicAction()
# # dyn.register(calculator, calculator_schema, required_role="any")
# # dyn.register(some_deploy_fn, deploy_schema, required_role="admin",
# #              precondition="run_tests")
# #
# # # As a regular user, deploy should NOT appear:
# # print(dyn.get_schemas(user_role="support"))  # only calculator
# #
# # # As admin before tests, deploy still hidden:
# # print(dyn.get_schemas(user_role="admin"))     # only calculator
# #
# # # After run_tests succeeds, deploy appears for admin:
# # dyn._completed_tools.add("run_tests")
# # print(dyn.get_schemas(user_role="admin"))     # calculator + deploy

print("Exercise 3: Fill in the TODOs above and uncomment to test.")

---

## Key Takeaways

| Pillar | Role | What breaks without it |
|--------|------|----------------------|
| **Perception** | Structures raw input for the LLM | Messy/ambiguous input leads to wrong tool calls and misclassified intent |
| **Memory** | Provides continuity across turns and sessions | Agent cannot follow multi-turn instructions or recall prior context |
| **Reasoning** | LLM-based decision making — the "brain" | No decisions get made at all (this pillar is never truly absent) |
| **Action** | Tool belt + validation + execution | Agent can only talk — it cannot *do* anything in the world |

> **The fundamental lesson:** An LLM is a reasoning engine, not a complete agent.
> The other components are not optional accessories — they are structural,
> load-bearing walls. (Chapter 3)